# Laxman MyTTS — Nepali Voice Cloning V1 🇳🇵

Generate Nepali speech using **your own reference voice** stored in this GitHub repository.

**Engine:** `Oshara/xtts-v2-nepali` (XTTS-v2 Nepali fine-tune)  
**Reference:** `voice/my sample.wav` from `LaxmanNepal/mytts`

> Use only your own voice or a voice for which you have permission.


In [ ]:
# 1) Install dependencies
!pip -q install -U coqui-tts==0.27.5 huggingface_hub gradio requests soundfile
print("Installed.")


In [ ]:
# 2) Check runtime
import torch, os
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU mode will be much slower. Use a Colab GPU runtime.")


In [ ]:
# 3) Download your voice sample from GitHub
import requests, pathlib

VOICE_URL = "https://github.com/LaxmanNepal/mytts/raw/refs/heads/main/voice/my%20sample.wav"
VOICE_PATH = "/content/laxman_reference.wav"

r = requests.get(VOICE_URL, timeout=60)
r.raise_for_status()
pathlib.Path(VOICE_PATH).write_bytes(r.content)

print("Reference voice saved:", VOICE_PATH)
print("Bytes:", len(r.content))


In [ ]:
# 4) Load the Nepali XTTS-v2 fine-tune
from huggingface_hub import snapshot_download
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

MODEL_ID = "Oshara/xtts-v2-nepali"
MODEL_SUBDIR = "epoch-10"

model_root = snapshot_download(
    MODEL_ID,
    allow_patterns=[f"{MODEL_SUBDIR}/*"]
)
MODEL_DIR = f"{model_root}/{MODEL_SUBDIR}"

config = XttsConfig()
config.load_json(f"{MODEL_DIR}/config.json")

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path=f"{MODEL_DIR}/model.pth",
    vocab_path=f"{MODEL_DIR}/vocab.json",
    speaker_file_path=f"{MODEL_DIR}/speakers_xtts.pth",
    eval=True,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    model.cuda()
else:
    model.cpu()

print("Model loaded on:", DEVICE)


In [ ]:
# 5) Reference-audio check + preview
import torchaudio
from IPython.display import Audio, display

wav, sr = torchaudio.load(VOICE_PATH)
duration = wav.shape[-1] / sr

print(f"Sample rate: {sr} Hz")
print(f"Channels: {wav.shape[0]}")
print(f"Duration: {duration:.2f} seconds")
display(Audio(VOICE_PATH))


In [ ]:
# 6) Nepali text normalisation + sentence chunking
import re

def normalise_nepali(text):
    text = text.replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"\s+", " ", text).strip()

def split_text(text, max_chars=240):
    text = normalise_nepali(text)
    if len(text) <= max_chars:
        return [text]
    sentences = re.split(r"(?<=[।!?])\s+", text)
    chunks, current = [], ""
    for sentence in sentences:
        if not sentence:
            continue
        if current and len(current) + len(sentence) + 1 > max_chars:
            chunks.append(current.strip())
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current:
        chunks.append(current.strip())
    return chunks

print(split_text("नमस्ते। म लक्ष्मण नेपाल हुँ। यो मेरो आफ्नै आवाजमा बनाइएको परीक्षण हो।"))


In [ ]:
# 7) Generate cloned Nepali speech
import torch
import torchaudio
from IPython.display import Audio, display

def generate_nepali(text, temperature=0.65, repetition_penalty=5.0,
                    output_path="/content/laxman_tts.wav"):
    chunks = split_text(text)
    outputs = []

    for chunk in chunks:
        result = model.synthesize(
            chunk,
            config,
            speaker_wav=VOICE_PATH,
            language="ne",
            temperature=float(temperature),
            repetition_penalty=float(repetition_penalty),
        )
        outputs.append(torch.tensor(result["wav"]).float())

    gap = torch.zeros(int(24000 * 0.18))
    pieces = []
    for i, audio in enumerate(outputs):
        pieces.append(audio)
        if i < len(outputs) - 1:
            pieces.append(gap)

    final = torch.cat(pieces).unsqueeze(0)
    torchaudio.save(output_path, final, 24000)

    display(Audio(output_path))
    return output_path

text = "नमस्ते, म लक्ष्मण नेपाल हुँ। यो मेरो आफ्नै आवाजमा तयार गरिएको नेपाली टेक्स्ट टु स्पिच परीक्षण हो।"
generate_nepali(text)


In [ ]:
# 8) Optional Gradio Studio UI
import gradio as gr

def gradio_generate(text, temperature, repetition_penalty):
    return generate_nepali(
        text,
        temperature=temperature,
        repetition_penalty=repetition_penalty,
        output_path="/content/laxman_gradio.wav",
    )

demo = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(
            label="नेपाली Text",
            lines=8,
            value="नमस्ते। म लक्ष्मण नेपाल हुँ। आज हामी एउटा नयाँ प्रविधिको बारेमा कुरा गर्दैछौँ।"
        ),
        gr.Slider(0.45, 0.90, value=0.65, step=0.05, label="Temperature"),
        gr.Slider(1.0, 8.0, value=5.0, step=0.5, label="Repetition penalty"),
    ],
    outputs=gr.Audio(label="Generated Voice", type="filepath"),
    title="Laxman MyTTS — Nepali Voice Studio",
    description="Your GitHub reference voice + Nepali XTTS-v2 voice cloning",
)

demo.launch(share=True)


## Tips

- A clean 3–10+ second reference with one speaker, little noise, no music and no clipping generally gives better cloning.
- For long narration, generate in sentence-sized chunks rather than one huge paragraph.
- If pronunciation becomes unstable, try shorter sentences and a slightly lower temperature.
- Keep this project for authorized voice cloning only.
